# NCKH — RF-DETR Medium + tracking/Re-ID
Bật Internet và T4 ×2. Run All một lần, giữ phiên hoạt động rồi gửi video từ web. Hiện dùng một GPU; không tự chia sang GPU thứ hai.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bản thử màu quần áo, giữ tracking và Re-ID hiện tại.
repo = Path('/kaggle/working/NCKH-colors')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'codex/kaggle-clothing-colors', 'https://github.com/Tiens0710/NCKH.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'backend' / 'requirements-rfdetr.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '--no-deps', 'git+https://github.com/KaiyangZhou/deep-person-reid.git'], check=True)
import cv2, torch, torchvision
from PIL import Image
code_revision = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Mã:', code_revision, '| PyTorch:', torch.__version__, '| TorchVision:', torchvision.__version__)
print('Thiết bị:', 'CUDA' if torch.cuda.is_available() else 'CPU')

In [ ]:
from getpass import getpass
from kaggle_secrets import UserSecretsClient

try:
    secret = UserSecretsClient().get_secret('SUPABASE_SECRET_KEY')
except Exception:
    secret = None
if not secret:
    secret = getpass('Dán SUPABASE_SECRET_KEY rồi nhấn Enter (ký tự sẽ được ẩn): ').strip()
if not secret or not secret.startswith('sb_secret_'):
    raise RuntimeError('Khóa không hợp lệ. Lấy sb_secret_... trong Supabase Settings → API Keys.')
os.environ['SUPABASE_URL'] = 'https://wtwjsisqghrqtqhzepci.supabase.co'
os.environ['SUPABASE_SECRET_KEY'] = secret
del secret
print('Đã nạp khóa Supabase cho phiên chạy này (không hiển thị giá trị).')

In [ ]:
from supabase import create_client

client = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SECRET_KEY'])
client.table('videos').select('id').limit(1).execute()
print('Kết nối Supabase OK. Khởi động worker ở ô tiếp theo; sau đó không cần chạy Kaggle lại mỗi lần thêm video.')

In [ ]:
# Cấu hình cho những video MỚI. max_frames=0 xử lý toàn video nhưng tốn thêm thời gian/GPU.
score_threshold = 0.4
sample_seconds = 0.5
max_frames = 0
# FP16 chỉ bật sau khi kiểm tra độ chính xác; FP32 là mặc định an toàn.
rfdetr_precision = 'fp32'
max_image_side = 1280
# Khởi động worker nền một lần. Worker tiếp tục chờ video mới mỗi 10 giây.
worker_log_path = Path('/kaggle/working/rfdetr-worker.log')
worker_process = globals().get('worker_process')
if worker_process is not None and worker_process.poll() is None and globals().get('worker_code_revision') != code_revision:
    print('Mã mới đã tải; chờ worker cũ hoàn tất video đang xử lý rồi khởi động lại...')
    worker_process.terminate()
    worker_process.wait(timeout=600)
if worker_process is None or worker_process.poll() is not None:
    worker_env = os.environ.copy()
    worker_env['PYTHONUNBUFFERED'] = '1'
    worker_env['NCKH_CODE_REVISION'] = code_revision
    worker_log_handle = worker_log_path.open('a', encoding='utf-8', buffering=1)
    worker_process = subprocess.Popen(
        [sys.executable, '-m', 'backend.ai.worker', '--detector', 'rfdetr-medium', '--rfdetr-precision', rfdetr_precision, '--enable-reid', '--backfill-latest', '--poll-seconds', '10', '--score-threshold', str(score_threshold), '--sample-seconds', str(sample_seconds), '--max-frames', str(max_frames), '--max-image-side', str(max_image_side), '--device', 'auto'],
        cwd=repo, env=worker_env, stdout=worker_log_handle, stderr=subprocess.STDOUT, start_new_session=True
    )
    worker_code_revision = code_revision
    print('Worker đang chạy nền. PID:', worker_process.pid)
else:
    print('Worker đã chạy rồi. PID:', worker_process.pid, '— nếu mới cập nhật mã, dừng worker cũ rồi chạy lại ô này.')
print('Giữ phiên Kaggle hoạt động; bây giờ có thể chuyển sang web và gửi video.')

In [ ]:
print('Trạng thái worker:', 'đang chạy' if worker_process.poll() is None else f'đã dừng (exit {worker_process.returncode})')
if worker_log_path.exists():
    log_text = worker_log_path.read_text(encoding='utf-8', errors='replace')
    print('--- Log worker mới nhất ---')
    print(log_text[-5000:] if log_text else '(chưa có log)')
rows = client.table('videos').select('status,metadata').order('created_at', desc=True).limit(20).execute().data
for row in rows:
    detection = (row.get('metadata') or {}).get('detection') or {}
    coverage = f"đến {detection.get('last_sample_time_seconds', '?')}s"
    if detection.get('video_duration_seconds') is not None:
        coverage += f" / {detection['video_duration_seconds']}s"
    if detection.get('truncated'):
        coverage += ' (CHƯA QUÉT HẾT VIDEO)'
    print(row['status'], '|', detection.get('device', '—'), '|', detection.get('person_boxes', 0), 'khung bao |', coverage)
print('Có video mới thì gửi từ web; worker tự nhận. Chạy lại ô này để xem log và kết quả cập nhật.')